# 01 Initial ETL
## Context
- I downloaded 365 csv files containing bus event data (arrival time, departure time etc.), which took up more than 60GB. 
- Data source (paid): https://tdx.transportdata.tw/
## Goal
- Identify corrupted files and inspect the data correctness
- Convert them to parquet format with a pecified schema to make future data wrangling efficient and easy.


In [3]:
import polars as pl
from constants import DATA_FOLDER, DATA_FILE

C:\Code\Python\Projects\04-BusTime\data\bus_event_time.parquet


# Loading csv files

In [ ]:
# At the time, there were 365 csv files sitting in the data folder. (and only the csv files, nothing else)
files = [f.name for f in DATA_FOLDER.iterdir() if f.is_file()]
cols = pl.scan_csv(DATA_FOLDER / files[0]).collect_schema().names()

In [ ]:
# check if they all had the same headers
for file in files:
    if cols != pl.scan_csv(DATA_FOLDER / file).collect_schema().names():
        print(f"{file} has a different header.")

Every csv has the same header.

In [ ]:
df = pl.read_csv(DATA_FOLDER / files[1], ignore_errors=True)
df.glimpse()

Rows: 529710
Columns: 29
$ PlateNumb         <str> 'KKA-3975', 'KKA-3975', 'KKA-3975', 'KKA-3975', 'KKA-3975', '359-FQ', '359-FQ', 'KKA-3975', 'KKA-3975', 'KKA-3975'
$ OperatorID        <i64> 16, 16, 16, 16, 16, 35, 35, 16, 16, 16
$ OperatorNo        <i64> 702, 702, 702, 702, 702, 1308, 1308, 702, 702, 702
$ RouteUID          <str> 'THB0968', 'THB0968', 'THB0968', 'THB0968', 'THB0968', 'THB0968', 'THB0968', 'THB0968', 'THB0968', 'THB0968'
$ RouteID           <i64> 968, 968, 968, 968, 968, 968, 968, 968, 968, 968
$ RouteNameZh_tw    <i64> 968, 968, 968, 968, 968, 968, 968, 968, 968, 968
$ RouteNameEn       <i64> 968, 968, 968, 968, 968, 968, 968, 968, 968, 968
$ SubRouteUID       <str> 'THB096802', 'THB096802', 'THB096802', 'THB096802', 'THB096801', 'THB096801', 'THB096801', 'THB096802', 'THB096802', 'THB096802'
$ SubRouteID        <i64> 96802, 96802, 96802, 96802, 96801, 96801, 96801, 96802, 96802, 96802
$ SubRouteNameZh_tw <i64> 968, 968, 968, 968, 968, 968, 968, 968, 968, 968
$ SubRo

In [ ]:
# Using pl.Categorical instead of pl.String for some columns for optimized storage
schema = {
    "PlateNumb": pl.String,
    "OperatorID": pl.Int32,
    "OperatorNo": pl.Int32,
    "RouteUID": pl.String,
    "RouteID": pl.Int32,
    "RouteNameZh_tw": pl.String,
    "RouteNameEn": pl.String,
    "SubRouteUID": pl.String,
    "SubRouteID": pl.String,
    "SubRouteNameZh_tw": pl.String,
    "SubRouteNameEn": pl.String,
    "Direction": pl.Categorical,
    "StopUID": pl.String,
    "StopID": pl.Int32,
    "StopNameZh_tw": pl.String,
    "StopNameEn": pl.String,
    "StopSequence": pl.Int32,
    "MessageType": pl.Categorical,
    "DutyStatus": pl.Categorical,
    "BusStatus": pl.Categorical,
    "A2EventType": pl.Categorical,
    "GPSTime": pl.String,
    "TripStartTimeType": pl.Categorical,
    "TripStartTime": pl.String,
    "TransTime": pl.String,
    "SrcRecTime": pl.String,
    "SrcTransTime": pl.String,
    "SrcUpdateTime": pl.String,
    "UpdateTime": pl.String,
}

In [ ]:
# Identify the corrupted csv files

error_files = []
for file in files:
    try:
        df = (
            pl.scan_csv(
                DATA_FOLDER / file,
                schema=schema,
            )
            .null_count()
            .collect()
        )
    except Exception as e:
        error_files.append(file)

print(error_files)

['公路客運定點歷史資料(A2)2025-03-08.CSV', '公路客運定點歷史資料(A2)2025-04-05.CSV', '公路客運定點歷史資料(A2)2025-05-12.CSV', '公路客運定點歷史資料(A2)2025-05-21.CSV', '公路客運定點歷史資料(A2)2025-06-03.CSV', '公路客運定點歷史資料(A2)2025-06-07.CSV', '公路客運定點歷史資料(A2)2025-06-12.CSV']


## Finding
- 7 out of 365 (1.9%) are corrupted. 
- After inspection, these errors aren't easy to fix and the proportion of corrupted files are too little that they can just be abandoned at this stage.

# Convert to parquet
- Converting to parquet allows extremely efficient data manipulation with polars later on
- Combine every csv into a single csv allows more efficient space use and easier query later on

In [ ]:
# Convert the non-corrupted csv files
correct_files = [DATA_FOLDER / file for file in files if file not in error_files]

pl.scan_csv(
    correct_files,
    schema=schema,
).sink_parquet(DATA_FILE)

In [5]:
# Check/Load the parquet file
df = pl.scan_parquet(DATA_FILE)
target = (
    df.drop(
        [
            "OperatorID",
            "OperatorNo",
            "RouteUID",
            "OperatorID",
            "OperatorNo",
            "RouteUID",
            "StopUID",
            "StopID",
            "TripStartTime",
            "TransTime",
            "SrcRecTime",
            "SrcTransTime",
            "SrcUpdateTime",
            "UpdateTime",
            "RouteNameZh_tw",
            "RouteNameEn",
            "SubRouteUID",
            "SubRouteID",
            "SubRouteNameZh_tw",
            "SubRouteNameEn",
            "StopNameEn",
            "MessageType",
            "TripStartTimeType",
        ]
    )
    .filter(
        pl.col("RouteID") == 1728,
        pl.col("StopNameZh_tw").is_in(["新竹轉運站", "龍潭運動公園"]),
        pl.col("A2EventType") == "離站",
    )
    .collect()
)
target.sample(20)

PlateNumb,RouteID,Direction,StopNameZh_tw,StopSequence,DutyStatus,BusStatus,A2EventType,GPSTime
str,i32,cat,str,i32,cat,cat,cat,str
"""KKB-2057""",1728,"""去程""","""龍潭運動公園""",15,"""開始""","""正常""","""離站""","""2026-01-22T20:34:02+08:00"""
"""KKB-2059""",1728,"""去程""","""龍潭運動公園""",15,"""開始""","""正常""","""離站""","""2025-03-12T06:39:02+08:00"""
"""KKB-2035""",1728,"""返程""","""新竹轉運站""",1,"""開始""","""正常""","""離站""","""2025-04-10T05:41:00+08:00"""
"""KKB-1715""",1728,"""返程""","""龍潭運動公園""",9,"""開始""","""正常""","""離站""","""2026-02-15T07:47:13+08:00"""
"""KKB-2057""",1728,"""返程""","""龍潭運動公園""",9,"""開始""","""正常""","""離站""","""2026-01-08T11:51:07+08:00"""
…,…,…,…,…,…,…,…,…
"""057-FS""",1728,"""返程""","""龍潭運動公園""",9,"""開始""","""正常""","""離站""","""2026-01-11T08:40:15+08:00"""
"""KKB-2056""",1728,"""返程""","""新竹轉運站""",1,"""開始""","""正常""","""離站""","""2025-12-21T15:01:36+08:00"""
"""KKB-2057""",1728,"""去程""","""龍潭運動公園""",15,"""開始""","""正常""","""離站""","""2025-05-04T12:10:12+08:00"""


## Finding
- Everything works
- The data storage size went from 60GB+ (for 365 csv files) to 4.9GB (for a single paraquet file)

In [ ]:
# Delete the csv files
for f in DATA_FOLDER.glob("*.csv"):
    f.unlink()